# Prediksi Selisih Tenaga Medis per Kabupaten/Kota di Sulawesi Tenggara

**Tujuan:** memprediksi selisih antara jumlah tenaga medis aktual dan tenaga medis ideal
(standar 0,8 tenaga medis per 1.000 penduduk) pada tahun-tahun mendatang.

**Input prediksi:** hanya *nama wilayah* dan *tahun*.

**Metode:** Regresi Linear, satu model per wilayah (agar bisa memproyeksikan tren ke tahun di luar data latih).

**Alur:**
1. Upload & baca data
2. Preprocessing
3. Eksplorasi data
4. Menghitung tenaga medis ideal dan selisih
5. Pembagian data latih/uji
6. Modeling
7. Evaluasi (dibandingkan dengan baseline)
8. Visualisasi tren per wilayah
9. Prediksi tahun ke depan

## 1. Upload & Baca Data

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Memakai nama file yang baru diunggah (aman walau Colab menambahkan "(1)" pada nama)
nama_file = list(uploaded.keys())[0]

# Jumlah_Penduduk dibaca sebagai teks agar angka seperti "162.780" tidak menjadi 162,78
df = pd.read_csv(nama_file, sep=';', dtype={'Jumlah_Penduduk': str})

print("=== DATASET ===")
display(df.head())
print("\nUkuran dataset:", df.shape)

## 2. Preprocessing

Pada kolom `Jumlah_Penduduk`, titik adalah pemisah ribuan (format Indonesia). Titik dihapus lalu kolom diubah menjadi bilangan bulat.
Kolom `Kabupaten_Kota` **tidak** di-one-hot encoding karena model dilatih terpisah untuk setiap wilayah.

In [ ]:
print("=== TIPE DATA SEBELUM ===")
print(df.dtypes)

df['Jumlah_Penduduk'] = (
    df['Jumlah_Penduduk']
    .str.replace('.', '', regex=False)
    .astype(int)
)

print("\n=== TIPE DATA SETELAH ===")
print(df.dtypes)

print("\n=== CEK HASIL PERBAIKAN ===")
display(df[['Kabupaten_Kota', 'Tahun', 'Jumlah_Penduduk']].head(10))

## 3. Eksplorasi Data

In [ ]:
print("=== INFORMASI DATA ===")
df.info()

print("\n=== STATISTIK DESKRIPTIF ===")
display(df.describe())

print("\n=== MISSING VALUE ===")
print(df.isnull().sum())

print("\n=== DATA DUPLIKAT ===")
print(df.duplicated().sum())

print("\n=== JUMLAH DATA PER WILAYAH ===")
print(df['Kabupaten_Kota'].value_counts().sort_index())

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df['Jumlah_Tenaga_Medis'], bins=10, edgecolor='black')
plt.title('Distribusi Jumlah Tenaga Medis')
plt.xlabel('Jumlah Tenaga Medis')
plt.ylabel('Jumlah Data')
plt.show()

## 4. Menghitung Tenaga Medis Ideal dan Selisih

- **Tenaga medis ideal** = jumlah penduduk × 0,8 ÷ 1000
- **Selisih** = tenaga medis aktual − tenaga medis ideal
  (positif = kelebihan, negatif = kekurangan). Kolom inilah yang menjadi **target** prediksi.

In [ ]:
standar_tenaga_medis = 0.8   # per 1.000 penduduk

df['Tenaga_Medis_Ideal'] = df['Jumlah_Penduduk'] * standar_tenaga_medis / 1000
df['Selisih_Tenaga_Medis'] = df['Jumlah_Tenaga_Medis'] - df['Tenaga_Medis_Ideal']

df = df.sort_values(['Kabupaten_Kota', 'Tahun']).reset_index(drop=True)

print("=== HASIL PERHITUNGAN ===")
display(df[['Kabupaten_Kota', 'Tahun', 'Jumlah_Penduduk', 'Jumlah_Tenaga_Medis',
            'Tenaga_Medis_Ideal', 'Selisih_Tenaga_Medis']].head(10))

## 5. Pembagian Data Latih dan Uji

Pembagian dilakukan **berdasarkan waktu** (bukan acak): data tahun awal untuk melatih, tahun terakhir untuk menguji.
Dengan beberapa tahun uji, terlihat apakah error membesar seiring jarak tahun prediksi.

In [ ]:
tahun_akhir_latih = 2022   # ubah sesuai kebutuhan

train = df[df['Tahun'] <= tahun_akhir_latih]
test  = df[df['Tahun'] >  tahun_akhir_latih]

print("=== DATA TRAINING ===")
print("Periode :", train['Tahun'].min(), "-", train['Tahun'].max())
print("Jumlah  :", len(train))

print("\n=== DATA TESTING ===")
print("Periode :", test['Tahun'].min(), "-", test['Tahun'].max())
print("Jumlah  :", len(test))

## 6. Modeling: Regresi Linear per Wilayah

Satu model regresi linear dilatih untuk tiap kabupaten/kota dengan fitur `Tahun` dan target `Selisih_Tenaga_Medis`.
Regresi linear membentuk garis tren sehingga dapat memprediksi tahun di luar rentang data latih
(berbeda dengan Random Forest yang hasilnya mendatar setelah tahun terakhir data latih).

In [ ]:
from sklearn.linear_model import LinearRegression

model_wilayah = {}
for wilayah, g in train.groupby('Kabupaten_Kota'):
    model_wilayah[wilayah] = LinearRegression().fit(g[['Tahun']], g['Selisih_Tenaga_Medis'])

def prediksi(wilayah, tahun):
    """Prediksi selisih tenaga medis untuk satu wilayah dan satu tahun."""
    return model_wilayah[wilayah].predict(pd.DataFrame({'Tahun': [tahun]}))[0]

print("Model berhasil dilatih untuk", len(model_wilayah), "wilayah.")

## 7. Evaluasi Model

Model dibandingkan dengan **baseline**: "selisih tahun depan sama dengan selisih tahun terakhir di data latih".
Model dianggap berguna jika errornya lebih kecil dari baseline.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_test = test['Selisih_Tenaga_Medis']

# Prediksi regresi linear
y_pred = test.apply(lambda r: prediksi(r['Kabupaten_Kota'], r['Tahun']), axis=1)

# Baseline: nilai tahun terakhir di data latih
terakhir = (
    train[train['Tahun'] == train['Tahun'].max()]
    .set_index('Kabupaten_Kota')['Selisih_Tenaga_Medis']
)
y_base = test['Kabupaten_Kota'].map(terakhir)

def evaluasi(nama, y_true, y_hat):
    print(f"=== {nama} ===")
    print("MAE  :", round(mean_absolute_error(y_true, y_hat), 3))
    print("RMSE :", round(np.sqrt(mean_squared_error(y_true, y_hat)), 3))
    print("R²   :", round(r2_score(y_true, y_hat), 4))
    print()

evaluasi("REGRESI LINEAR", y_test, y_pred)
evaluasi("BASELINE (sama seperti tahun terakhir)", y_test, y_base)

In [ ]:
hasil = pd.DataFrame({
    'Kabupaten_Kota': test['Kabupaten_Kota'].values,
    'Tahun': test['Tahun'].values,
    'Aktual': y_test.values,
    'Prediksi': y_pred.values,
    'Baseline': y_base.values
})
hasil['Error_Prediksi'] = (hasil['Aktual'] - hasil['Prediksi']).abs()

display(hasil)

In [ ]:
# Error rata-rata per tahun uji (apakah error membesar untuk tahun yang lebih jauh?)
err_tahun = hasil.groupby('Tahun').apply(
    lambda g: pd.Series({
        'MAE_Linear': (g['Aktual'] - g['Prediksi']).abs().mean(),
        'MAE_Baseline': (g['Aktual'] - g['Baseline']).abs().mean()
    })
)
display(err_tahun.round(2))

## 8. Visualisasi Tren per Wilayah

Grafik membantu memastikan apakah pola data memang cukup lurus untuk dimodelkan dengan regresi linear.
Titik biru = data latih, titik merah = data uji, garis = tren model.

In [ ]:
wilayah_list = sorted(df['Kabupaten_Kota'].unique())
n = len(wilayah_list)
kolom = 3
baris = int(np.ceil(n / kolom))

fig, axes = plt.subplots(baris, kolom, figsize=(15, 3.2 * baris), sharex=True)
axes = axes.flatten()

tahun_range = pd.DataFrame({'Tahun': np.arange(df['Tahun'].min(), df['Tahun'].max() + 1)})

for ax, w in zip(axes, wilayah_list):
    d = df[df['Kabupaten_Kota'] == w]
    d_tr = d[d['Tahun'] <= tahun_akhir_latih]
    d_te = d[d['Tahun'] >  tahun_akhir_latih]

    ax.scatter(d_tr['Tahun'], d_tr['Selisih_Tenaga_Medis'], color='tab:blue', s=20, label='Latih')
    ax.scatter(d_te['Tahun'], d_te['Selisih_Tenaga_Medis'], color='tab:red', s=20, label='Uji')
    ax.plot(tahun_range['Tahun'], model_wilayah[w].predict(tahun_range), color='black', lw=1, label='Tren')
    ax.axhline(0, color='gray', lw=0.5, ls='--')
    ax.set_title(w, fontsize=10)

for ax in axes[n:]:
    ax.axis('off')

axes[0].legend(fontsize=8)
fig.suptitle('Selisih Tenaga Medis per Wilayah: Aktual vs Tren Linear', y=1.0)
plt.tight_layout()
plt.show()

## 9. Prediksi Tahun ke Depan

Setelah evaluasi, model **dilatih ulang memakai seluruh data** agar tahun terbaru ikut dipelajari.
Input prediksi hanya **nama wilayah dan tahun**.

In [ ]:
model_final = {}
for wilayah, g in df.groupby('Kabupaten_Kota'):
    model_final[wilayah] = LinearRegression().fit(g[['Tahun']], g['Selisih_Tenaga_Medis'])

def prediksi_final(wilayah, tahun):
    """Prediksi selisih tenaga medis (positif = kelebihan, negatif = kekurangan)."""
    return model_final[wilayah].predict(pd.DataFrame({'Tahun': [tahun]}))[0]

# Contoh penggunaan untuk satu wilayah
contoh_wilayah = 'Kendari'
contoh_tahun = 2028
if contoh_wilayah in model_final:
    print(f"Prediksi selisih tenaga medis {contoh_wilayah} tahun {contoh_tahun}: "
          f"{prediksi_final(contoh_wilayah, contoh_tahun):.1f} orang")

In [ ]:
tahun_prediksi = [2026, 2027, 2028, 2029, 2030]   # ubah sesuai kebutuhan

ramalan = pd.DataFrame([
    {'Kabupaten_Kota': w, 'Tahun': t, 'Prediksi_Selisih': round(prediksi_final(w, t), 1)}
    for w in sorted(model_final) for t in tahun_prediksi
])

# Tabel ringkas: baris = wilayah, kolom = tahun
tabel_ramalan = ramalan.pivot(index='Kabupaten_Kota', columns='Tahun', values='Prediksi_Selisih')
display(tabel_ramalan)

In [ ]:
# (Opsional) simpan hasil prediksi ke CSV lalu unduh
tabel_ramalan.to_csv('prediksi_selisih_tenaga_medis.csv', sep=';')
files.download('prediksi_selisih_tenaga_medis.csv')

## Catatan Interpretasi

- Nilai **positif** = jumlah tenaga medis melebihi standar; nilai **negatif** = kekurangan tenaga medis.
- Prediksi memakai garis tren linear, sehingga semakin jauh dari tahun terakhir data, semakin besar ketidakpastiannya. Disarankan membatasi horizon prediksi (misalnya 3 sampai 5 tahun).
- Setiap wilayah hanya memiliki sekitar 11 titik data, sehingga model sederhana lebih aman daripada model kompleks.
- Jika grafik pada bagian 8 menunjukkan pola yang melengkung atau melompat, pertimbangkan model lain (misalnya Holt/exponential smoothing atau hybrid linear + Random Forest).